# Научный клубок — Норникель
## Поисково-аналитическая система знаний R&D

**Возможности:**
- Загрузка PDF и DOCX из любой вложенной структуры папок
- Семантический поиск по корпусу документов
- Аналитические ответы через российские LLM (GigaChat / YandexGPT / Mistral)
- Граф связей: материалы → процессы → оборудование → результаты
- Детектор пробелов в исследованиях
- Фильтрация по типу документа, источнику, году, географии


In [ ]:
# Установка библиотек

!pip install chromadb pdfplumber python-docx beautifulsoup4 requests networkx pyvis ipywidgets pymupdf --quiet

print("Библиотеки готовы")

In [ ]:
# Импорты

import json
import logging
import os
import re
import zipfile
from collections import Counter
from datetime import datetime
from pathlib import Path

import chromadb
import ipywidgets as widgets
import networkx as nx
import requests as _req
from IPython.display import HTML, clear_output, display
from chromadb.utils import embedding_functions
from docx import Document as DocxDocument
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Inches, Pt, RGBColor
from google.colab import files
from pyvis.network import Network
from tqdm import tqdm

print("Импорты выполнены")

In [ ]:
# # с яндекс драйва

# EXTS = {'.pdf', '.docx', '.doc', '.txt', '.xlsx'}

# stats = {'total': 0, 'skip_ext': 0, 'skip_exists': 0, 'ok': 0, 'errors': 0}

# def download_tree(path='/'):
#     for it in list_dir(path):
#         if it['type'] == 'dir':
#             download_tree(it['path'])
#             continue

#         stats['total'] += 1
#         ext = Path(it['name']).suffix.lower()
#         if ext not in EXTS:
#             stats['skip_ext'] += 1
#             continue

#         local = Path(DATA_DIR) / it['path'].lstrip('/')
#         local.parent.mkdir(parents=True, exist_ok=True)
#         if local.exists():
#             stats['skip_exists'] += 1
#             continue

#         for attempt in range(3):
#             try:
#                 href = requests.get(API + '/download',
#                                     params={'public_key': PUBLIC_URL, 'path': it['path']},
#                                     timeout=30).json()['href']
#                 with requests.get(href, stream=True, timeout=180) as r:  # таймаут побольше для крупных файлов
#                     r.raise_for_status()
#                     with open(local, 'wb') as f:
#                         for chunk in r.iter_content(1 << 20):
#                             f.write(chunk)
#                 stats['ok'] += 1
#                 break
#             except Exception as e:
#                 if attempt == 2:
#                     stats['errors'] += 1
#                     print(f"  ОШИБКА {it['path']}: {e}")
#                 else:
#                     time.sleep(1.5 * (attempt + 1))

# print('Начинаем обход всего диска...')
# download_tree('/')

# print('\nИтого:', stats)
# n = sum(1 for _ in Path(DATA_DIR).rglob('*') if _.is_file())
# print(f'Файлов скачано в {DATA_DIR}: {n}')

In [ ]:
# Загрузка данных с Google Drive

# ВАЖНО: не публикуйте здесь ID реального Google Drive файла с внутренними документами.
# Укажите свой ID через переменную окружения либо впишите вручную перед запуском.
FILE_ID = os.environ.get('NORNICKEL_DATA_FILE_ID', 'ВСТАВЬТЕ_СВОЙ_FILE_ID')
ZIP_NAME = '/content/data.zip'
DATA_DIR = '/content/data'

print('Скачиваем архив с Google Drive...')
os.system(f'gdown --id {FILE_ID} -O {ZIP_NAME}')

if not os.path.exists(ZIP_NAME):
    os.system('pip install gdown --quiet')
    os.system(f'gdown --id {FILE_ID} -O {ZIP_NAME}')

os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_NAME, 'r') as z:
    z.extractall(DATA_DIR)
print(f'Архив распакован в {DATA_DIR}')

subdirs = [d for d in Path(DATA_DIR).iterdir() if d.is_dir()]
if len(subdirs) == 1:
    DATA_DIR = str(subdirs[0])
    print(f'Корневая папка: {DATA_DIR}')

all_files = list(Path(DATA_DIR).rglob('*'))
pdf_n  = sum(1 for f in all_files if f.suffix.lower() == '.pdf')
docx_n = sum(1 for f in all_files if f.suffix.lower() in ('.docx', '.doc'))
txt_n  = sum(1 for f in all_files if f.suffix.lower() == '.txt')
print(f'\nНайдено файлов: PDF={pdf_n}, DOCX={docx_n}, TXT={txt_n}')
print(f'Всего объектов в дереве: {len(all_files)}')
print('\nПервые 15 файлов:')
for f in sorted(all_files)[:15]:
    if f.is_file():
        print(f'  {f.relative_to(DATA_DIR)}')

In [ ]:
# Парсинг документов (кеш в SQLite + ограничение объёма чтения + параллельная обработка)

import sqlite3
from concurrent.futures import ProcessPoolExecutor, as_completed

logging.getLogger('pdfminer').setLevel(logging.ERROR)

MAX_CHUNKS_PER_DOC = 15       # не более 15 чанков с одного документа
CHAR_BUDGET        = 8000     # столько символов достаточно на MAX_CHUNKS_PER_DOC чанков (с запасом)
SUPPORTED          = {'.pdf', '.docx', '.doc', '.txt'}
CACHE_DB           = str(Path(DATA_DIR).parent / 'parse_cache.sqlite')


# ---------- кеш на диске ----------

def init_cache(db_path):
    conn = sqlite3.connect(db_path)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS parsed_docs (
            filepath    TEXT PRIMARY KEY,   -- путь относительно DATA_DIR
            fingerprint TEXT NOT NULL,      -- mtime_ns:size, чтобы ловить изменения файла
            id          TEXT,
            filename    TEXT,
            doc_type    TEXT,
            source      TEXT,
            year        TEXT,
            geography   TEXT,
            text        TEXT,
            chunks      TEXT                -- json-список чанков
        )
    ''')
    conn.commit()
    return conn


def file_fingerprint(fpath: Path) -> str:
    st = fpath.stat()
    return f'{st.st_mtime_ns}:{st.st_size}'


def cache_get_all(conn):
    rows = conn.execute(
        'SELECT filepath, fingerprint, id, filename, doc_type, source, year, geography, text, chunks '
        'FROM parsed_docs'
    ).fetchall()
    return {
        r[0]: {
            'fingerprint': r[1], 'id': r[2], 'filename': r[3],
            'doc_type': r[4], 'source': r[5], 'year': r[6], 'geography': r[7],
            'text': r[8], 'chunks': json.loads(r[9]),
        }
        for r in rows
    }


def cache_upsert_many(conn, records):
    conn.executemany('''
        INSERT INTO parsed_docs
            (filepath, fingerprint, id, filename, doc_type, source, year, geography, text, chunks)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(filepath) DO UPDATE SET
            fingerprint = excluded.fingerprint, id = excluded.id, filename = excluded.filename,
            doc_type    = excluded.doc_type,    source = excluded.source,  year = excluded.year,
            geography   = excluded.geography,   text   = excluded.text,    chunks = excluded.chunks
    ''', records)
    conn.commit()


# ---------- парсеры (читают не больше CHAR_BUDGET символов) ----------

def parse_pdf(path, char_budget=CHAR_BUDGET):
    try:
        import fitz
        doc = fitz.open(path)
        parts, total = [], 0
        for page in doc:
            t = page.get_text()
            parts.append(t)
            total += len(t)
            if total >= char_budget:
                break
        doc.close()
        return '\n'.join(parts)
    except Exception:
        return ''


def parse_docx(path, char_budget=CHAR_BUDGET):
    try:
        from docx import Document
        doc = Document(path)
        parts, total = [], 0
        for p in doc.paragraphs:
            if p.text.strip():
                parts.append(p.text)
                total += len(p.text)
                if total >= char_budget:
                    return '\n'.join(parts)
        # таблицы раньше игнорировались полностью — теряли параметры/данные из них
        for table in doc.tables:
            for row in table.rows:
                cells = [c.text.strip() for c in row.cells if c.text.strip()]
                if not cells:
                    continue
                line = ' | '.join(cells)
                parts.append(line)
                total += len(line)
                if total >= char_budget:
                    return '\n'.join(parts)
        return '\n'.join(parts)
    except Exception:
        return ''


def parse_file(path):
    ext = Path(path).suffix.lower()
    if ext == '.pdf':
        return parse_pdf(path)
    elif ext in ('.docx', '.doc'):
        return parse_docx(path)
    elif ext == '.txt':
        try:
            with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                return f.read(CHAR_BUDGET)
        except Exception:
            return ''
    return ''


def extract_meta_from_path(fpath, base_dir):
    parts = list(Path(fpath).relative_to(base_dir).parts)[:-1]
    doc_type = parts[0] if len(parts) >= 1 else 'Не указано'
    source   = parts[1] if len(parts) >= 2 else 'Не указано'
    year = 'Не указано'
    for part in parts:
        m = re.match(r'^((?:19|20)\d{2})(-\d+)?$', part)
        if m:
            year = m.group(1)
            break
    return doc_type, source, year


def detect_geography(text):
    t = text.lower()
    ru = sum(1 for kw in ['россия', 'российск', 'норильск', 'отечествен', 'норникель', 'рф'] if kw in t)
    en = sum(1 for kw in ['canada', 'finland', 'australia', 'usa', 'united states', 'зарубеж', 'мировая практика'] if kw in t)
    if ru > en:
        return 'Россия'
    elif en > 0:
        return 'Зарубежье'
    return 'Не указано'


def chunk_text(text, chunk_size=500, overlap=100):
    if not text or len(text) < 50:
        return []
    chunks, start = [], 0
    while start < len(text):
        chunk = text[start:start + chunk_size].strip()
        if len(chunk) > 50:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


def process_one(fpath_str, base_dir_str):
    """Выполняется в отдельном процессе — своего sqlite-соединения не открывает,
    только парсит файл и возвращает готовую запись."""
    fpath = Path(fpath_str)
    text = parse_file(fpath_str)
    if not text or len(text) < 50:
        return None
    doc_type, source, year = extract_meta_from_path(fpath, base_dir_str)
    if year == 'Не указано':
        years = re.findall(r'\b(20[0-2][0-9]|19[89][0-9])\b', text)
        year = Counter(years).most_common(1)[0][0] if years else 'Не указано'
    geo = detect_geography(text)
    chunks = chunk_text(text)[:MAX_CHUNKS_PER_DOC]
    rel = str(fpath.relative_to(base_dir_str))
    doc_id = rel.replace(os.sep, '_').replace(' ', '_')
    return {
        'filepath': rel, 'id': doc_id, 'filename': fpath.name,
        'doc_type': doc_type, 'source': source, 'year': year,
        'geography': geo, 'text': text, 'chunks': chunks,
    }


# ---------- основной цикл: сверяем с кешем, парсим только новое/изменённое ----------

conn = init_cache(CACHE_DB)
cached = cache_get_all(conn)

all_paths = [f for f in sorted(Path(DATA_DIR).rglob('*'))
             if f.is_file() and f.suffix.lower() in SUPPORTED]

to_parse = []
reused = 0
for fpath in all_paths:
    rel = str(fpath.relative_to(DATA_DIR))
    fp = file_fingerprint(fpath)
    hit = cached.get(rel)
    if hit and hit['fingerprint'] == fp:
        reused += 1
    else:
        to_parse.append((str(fpath), fp, rel))

print(f'Всего файлов: {len(all_paths)} | взято из кеша: {reused} | нужно распарсить: {len(to_parse)}')

if to_parse:
    new_records = []
    with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
        futures = {ex.submit(process_one, p, DATA_DIR): (fp, rel) for p, fp, rel in to_parse}
        for fut in tqdm(as_completed(futures), total=len(futures), desc='Парсинг'):
            fp, rel = futures[fut]
            res = fut.result()
            if res is None:
                continue
            new_records.append((
                rel, fp, res['id'], res['filename'], res['doc_type'],
                res['source'], res['year'], res['geography'],
                res['text'], json.dumps(res['chunks']),
            ))
    if new_records:
        cache_upsert_many(conn, new_records)
        cached = cache_get_all(conn)  # перечитываем кеш с учётом новых записей

# ---------- собираем DOCUMENTS из кеша (уже консистентно: и старое, и новое) ----------

DOCUMENTS = []
skipped = 0
for fpath in all_paths:
    rel = str(fpath.relative_to(DATA_DIR))
    row = cached.get(rel)
    if not row:
        skipped += 1
        continue
    DOCUMENTS.append({
        'id':        row['id'],
        'filename':  row['filename'],
        'filepath':  rel,
        'doc_type':  row['doc_type'],
        'source':    row['source'],
        'year':      row['year'],
        'geography': row['geography'],
        'text':      row['text'],
        'chunks':    row['chunks'],
    })

total_chunks = sum(len(d['chunks']) for d in DOCUMENTS)
print(f'\nИтого: {len(DOCUMENTS)} документов, {total_chunks} чанков, пропущено {skipped}')
print('По типам:', dict(Counter(d['doc_type'] for d in DOCUMENTS)))
print('По географии:', dict(Counter(d['geography'] for d in DOCUMENTS)))


In [ ]:
# Векторный поиск (ChromaDB)

chroma_client = chromadb.Client()
ef = embedding_functions.DefaultEmbeddingFunction()

try: chroma_client.delete_collection('nornickel')
except: pass
collection = chroma_client.create_collection('nornickel', embedding_function=ef)

docs_list, metas_list, ids_list = [], [], []
for doc in DOCUMENTS:
    for i, chunk in enumerate(doc['chunks']):
        docs_list.append(chunk)
        metas_list.append({
            'filename':  doc['filename'],
            'filepath':  doc['filepath'],
            'doc_type':  doc['doc_type'],
            'source':    doc['source'],
            'year':      doc['year'],
            'geography': doc['geography'],
        })
        ids_list.append(f"{doc['id']}_c{i}")

print(f'Чанков для индексации: {len(docs_list)}')
print('Начинаем индексацию...')

# Первый батч отдельно — он скачает модель, остальные быстрее
print('Шаг 1: скачиваем модель (первый батч)...')
collection.add(
    documents=docs_list[:100],
    metadatas=metas_list[:100],
    ids=ids_list[:100]
)
print(f'Модель загружена, в базе: {collection.count()} чанков')

# Остальные батчи с прогресс-баром
print('Шаг 2: индексируем оставшиеся чанки...')
for i in tqdm(range(100, len(docs_list), 500)):
    collection.add(
        documents=docs_list[i:i+500],
        metadatas=metas_list[i:i+500],
        ids=ids_list[i:i+500]
    )

print(f'\nПроиндексировано: {collection.count()} чанков из {len(DOCUMENTS)} документов')


def search(query, n_results=5, geography=None, doc_type=None, year=None):
    where = {}
    if geography: where['geography'] = geography
    if doc_type:  where['doc_type']  = doc_type
    if year:      where['year']      = year
    n = min(n_results, max(1, collection.count()))
    kwargs = dict(query_texts=[query], n_results=n)
    if where:
        kwargs['where'] = where if len(where) == 1 else {'$and': [{k: v} for k, v in where.items()]}
    try:
        res = collection.query(**kwargs)
    except Exception:
        res = collection.query(query_texts=[query], n_results=n)
    return [{
        'text':      txt,
        'filename':  meta.get('filename', ''),
        'filepath':  meta.get('filepath', ''),
        'doc_type':  meta.get('doc_type', ''),
        'source':    meta.get('source', ''),
        'year':      meta.get('year', ''),
        'geography': meta.get('geography', ''),
        'score':     round(res['distances'][0][i], 3),
    } for i, (txt, meta) in enumerate(zip(res['documents'][0], res['metadatas'][0]))]


print('\nТест поиска:')
for r in search('электроэкстракция никеля католит', n_results=2):
    print(f'  -> [{r["doc_type"]}][{r["year"]}] {r["filename"]} score={r["score"]}')

In [ ]:
# YANDEXGPT

import getpass

# Ключ и ID каталога не хранятся в коде.
# Вариант 1 (рекомендуется в Colab): Secrets -> добавить YANDEX_API_KEY и YANDEX_FOLDER,
# затем: from google.colab import userdata; YANDEX_API_KEY = userdata.get('YANDEX_API_KEY')
# Вариант 2: переменные окружения (для запуска не в Colab)
# Вариант 3: ручной ввод ниже (ключ не отображается при вводе и не попадает в файл)
YANDEX_API_KEY = os.environ.get('YANDEX_API_KEY') or getpass.getpass('Введите YANDEX_API_KEY: ')
YANDEX_FOLDER  = os.environ.get('YANDEX_FOLDER')  or input('Введите YANDEX_FOLDER: ')

def ask_llm(prompt):
    url = 'https://llm.api.cloud.yandex.net/foundationModels/v1/completion'
    headers = {
        'Authorization': f'Api-Key {YANDEX_API_KEY}',
        'Content-Type': 'application/json'
    }
    body = {
        'modelUri': f'gpt://{YANDEX_FOLDER}/yandexgpt-lite',
        'completionOptions': {
            'stream': False,
            'temperature': 0.3,
            'maxTokens': 1500
        },
        'messages': [{'role': 'user', 'text': prompt}]
    }
    try:
        r = _req.post(url, headers=headers, json=body, timeout=30)
        return r.json()['result']['alternatives'][0]['message']['text']
    except Exception as e:
        return f'[Ошибка YandexGPT: {e}]'

print('YandexGPT настроен — вставьте ключи выше')

In [ ]:
# Аналитические функции


def answer_question(query, geography=None, doc_type=None, year=None):
    """RAG: поиск релевантных фрагментов + аналитический ответ LLM"""
    results = search(query, n_results=6, geography=geography,
                     doc_type=doc_type, year=year)
    if not results:
        return 'По данному запросу информация в базе не найдена.'
    filters = []
    if geography: filters.append('география: ' + geography)
    if doc_type:  filters.append('тип: ' + doc_type)
    if year:      filters.append('год: ' + year)
    filter_label = ' [' + ', '.join(filters) + ']' if filters else ''
    context_parts = []
    for r in results:
        header = '[' + r['doc_type'] + ' / ' + r['source'] + ' / ' + r['year'] + ' / ' + r['geography'] + ']'
        context_parts.append(header + '\n' + r['text'])
    context = '\n\n'.join(context_parts)
    prompt = (
        'Ты — аналитик R&D системы компании Норникель.\n'
        'На основе найденных фрагментов дай структурированный аналитический ответ.\n\n'
        'Запрос' + filter_label + ': ' + query + '\n\n'
        'Источники:\n' + context + '\n\n'
        'Требования к ответу:\n'
        '1. Структурированный ответ по ключевым аспектам\n'
        '2. Укажи источник каждого факта (название файла / журнал / год)\n'
        '3. Выдели отечественную и зарубежную практику если есть различия\n'
        '4. Приведи числовые параметры если упомянуты\n'
        '5. Отметь противоречия между источниками\n'
        '6. Язык ответа: русский'
    )
    return ask_llm(prompt)


def compare_geo(query):
    """Сравнение отечественной и зарубежной практики по запросу."""
    ru   = search(query, n_results=4, geography='Россия')
    intl = search(query, n_results=4, geography='Зарубежье')
    ru_lines   = ['- [' + r['source'] + ', ' + r['year'] + ']: ' + r['text'][:300] for r in ru]
    intl_lines = ['- [' + r['source'] + ', ' + r['year'] + ']: ' + r['text'][:300] for r in intl]
    ru_ctx   = '\n'.join(ru_lines)   if ru_lines   else 'Нет данных'
    intl_ctx = '\n'.join(intl_lines) if intl_lines else 'Нет данных'
    prompt = (
        'Ты — аналитик R&D Норникеля.\n'
        'Тема: ' + query + '\n\n'
        'РОССИЯ:\n' + ru_ctx + '\n\n'
        'ЗАРУБЕЖНАЯ ПРАКТИКА:\n' + intl_ctx + '\n\n'
        'Сравни по пунктам:\n'
        '1. Ключевые различия в подходах\n'
        '2. Преимущества каждого подхода\n'
        '3. Применимость зарубежного опыта в российских условиях\n'
        '4. Вывод\n'
        'Язык: русский'
    )
    return ask_llm(prompt)


def detect_gaps():
    """Детектор пробелов: показывает непокрытые темы и статистику базы."""
    all_text = ' '.join(d['text'].lower() for d in DOCUMENTS)
    # Ключевые слова на русском — для реальных документов Норникеля
    topics = {
        'Электроэкстракция никеля':        ['электроэкстракция', 'electrowinning', 'католит', 'анолит'],
        'Кучное выщелачивание':            ['кучное выщелачивание', 'heap leaching', 'штабель'],
        'Обессоливание воды':              ['обессоливание', 'обратный осмос', 'ионный обмен', 'нанофильтрация'],
        'Закачка шахтных вод':             ['закачка', 'шахтных вод', 'injection well'],
        'Распределение МПГ / штейн-шлак':  ['мпг', 'штейн', 'шлак', 'золото', 'серебро'],
        'Печь взвешенной плавки (ПВП)':    ['пвп', 'взвешенной плавки', 'fluidized bed'],
        'Очистка газов SO2':               ['очистка газов', 'диоксид серы', 'скруббер'],
        'Переработка техногенного гипса':  ['техногенный гипс', 'гипс', 'переработка отходов'],
        'Гидрометаллургия меди':           ['гидрометаллургия', 'медь', 'медного'],
        'Флотация сульфидных руд':         ['флотация', 'сульфидн', 'флотационн'],
        'Нейтрализация кислых стоков':     ['нейтрализация', 'кислые стоки', 'amd'],
        'Переработка редкоземельных':      ['редкоземельн', 'рзм', 'rare earth'],
    }
    covered, gaps = [], []
    for topic, kws in topics.items():
        (covered if any(kw in all_text for kw in kws) else gaps).append(topic)
    type_stats = Counter(d['doc_type']  for d in DOCUMENTS)
    geo_stats  = Counter(d['geography'] for d in DOCUMENTS)
    year_stats = Counter(d['year']      for d in DOCUMENTS)
    lines = [
        'ПОКРЫТИЕ БАЗЫ ЗНАНИЙ', '',
        f'Документов: {len(DOCUMENTS)} | Чанков: {collection.count()}', '',
        'По типам документов:',
    ]
    for t, c in type_stats.most_common():
        lines.append(f'  {t}: {c}')
    lines += ['', 'По географии:']
    for g, c in geo_stats.most_common():
        lines.append(f'  {g}: {c}')
    lines += ['', 'По годам:']
    for y, c in sorted(year_stats.items()):
        lines.append(f'  {y}: {c}')
    lines += ['', f'Темы с покрытием ({len(covered)}/{len(topics)}):']
    for t in covered:
        lines.append(f'  [+] {t}')
    lines += ['', f'Пробелы — темы без данных ({len(gaps)}/{len(topics)}):']
    for t in gaps:
        lines.append(f'  [-] {t}')
    return '\n'.join(lines)

print('Аналитические функции готовы')

In [ ]:
# Граф связей

# Ключевые слова на русском для реальных документов
ENTITY_KEYWORDS = {
    'материал':     ['никель','медь','палладий','золото','серебро','мпг',
                     'гипс','штейн','шлак','сульфат','хлорид','электролит'],
    'процесс':      ['электроэкстракция','выщелачивание','флотация','плавка',
                     'обессоливание','закачка','нейтрализация','очистка'],
    'оборудование': ['ванна','печь','пвп','скруббер','мембрана','диафрагма','насос','фильтр'],
    'параметр':     ['температура','концентрация','скорость','давление','ph','плотность'],
}
NODE_COLORS = {
    'материал':     '#4E9AF1',
    'процесс':      '#F1A24E',
    'оборудование': '#6ABF69',
    'параметр':     '#E06B6B',
    'документ':     '#9B59B6',
}

def build_graph(query=None, max_docs=15):
    """
    Строит граф связей из корпуса документов.
    query — если указан, строит граф только по релевантным документам
    """
    G = nx.Graph()
    if query:
        results = search(query, n_results=max_docs)
        fnames = set(r['filename'] for r in results)
        docs_to_use = [d for d in DOCUMENTS if d['filename'] in fnames]
    else:
        docs_to_use = DOCUMENTS[:max_docs]

    for doc in docs_to_use:
        dnode = doc['doc_type'] + ': ' + doc['filename']
        if not G.has_node(dnode):
            G.add_node(dnode,
                       color=NODE_COLORS['документ'],
                       title=doc['filepath'] + '\n' + doc['geography'] + ', ' + doc['year'],
                       size=20)
        tl = doc['text'].lower()
        for etype, kws in ENTITY_KEYWORDS.items():
            for kw in kws:
                if kw in tl:
                    nid = etype + ':' + kw
                    if not G.has_node(nid):
                        G.add_node(nid,
                                   color=NODE_COLORS[etype],
                                   title=etype + ': ' + kw,
                                   label=kw,
                                   size=15)
                    G.add_edge(dnode, nid)

    net = Network(height='640px', width='100%', bgcolor='#1a1a2e',
                  font_color='white', notebook=True, cdn_resources='in_line')
    net.from_nx(G)
    net.set_options('{"physics":{"forceAtlas2Based":{"gravitationalConstant":-60,"springLength":140},"solver":"forceAtlas2Based"},"interaction":{"hover":true}}')
    title = 'по запросу: ' + query if query else 'весь корпус'
    print(f'Граф ({title}): {G.number_of_nodes()} узлов, {G.number_of_edges()} рёбер')
    print('Легенда: синий=Материал  оранжевый=Процесс  зелёный=Оборудование  красный=Параметр  фиолетовый=Документ')
    net.show('graph.html')
    display(HTML('graph.html'))

print('Граф готов')

In [ ]:
# Настройка интерфейса

CSS = (
    '<style>'
    '.card{background:#1e1e2e;border:1px solid #3a3a5c;border-radius:10px;'
    'padding:14px;margin:8px 0;color:#cdd6f4;}'
    '.tag{display:inline-block;background:#313244;border-radius:4px;'
    'padding:2px 8px;margin:2px;font-size:12px;}'
    '.ftitle{font-size:14px;font-weight:bold;color:#89b4fa;margin-bottom:6px;}'
    '.snippet{font-size:13px;color:#a6adc8;margin-top:6px;line-height:1.5;}'
    '.stitle{font-size:17px;font-weight:bold;color:#cba6f7;margin:16px 0 8px;}'
    '.llm{background:#1a2e1e;border:1px solid #3a7c4e;border-radius:10px;'
    'padding:16px;color:#a6e3a1;white-space:pre-wrap;line-height:1.6;}'
    '.gap{background:#2a1a2e;border:1px solid #6e3a7c;border-radius:10px;'
    'padding:16px;color:#f5c2e7;white-space:pre-wrap;font-family:monospace;}'
    '.cmp{background:#1a1e2e;border:1px solid #4e6a9c;border-radius:10px;'
    'padding:16px;color:#cdd6f4;white-space:pre-wrap;line-height:1.6;}'
    '</style>'
)

def render_card(r):
    snippet = r['text'][:300].replace('<','&lt;').replace('>','&gt;')
    return (
        '<div class="card">'
        '<div class="ftitle"> ' + r['filename'] + '</div>'
        '<div>'
        '<span class="tag">' + r['doc_type']  + '</span>'
        '<span class="tag">' + r['source']    + '</span>'
        '<span class="tag">' + r['year']      + '</span>'
        '<span class="tag">' + r['geography'] + '</span>'
        '<span class="tag">score: ' + str(r['score']) + '</span>'
        '</div>'
        '<div class="snippet">' + snippet + '...</div>'
        '</div>'
    )

# Фильтры заполняются из реальных данных
doc_types = ['Все типы']  + sorted(set(d['doc_type'] for d in DOCUMENTS))
geo_opts  = ['Все источники', 'Россия', 'Зарубежные']
year_opts = ['Все годы']  + sorted(set(d['year'] for d in DOCUMENTS if d['year'] != 'Не указано'))

st = {'description_width': '110px'}
query_inp = widgets.Text(
    placeholder='Например: обессоливание воды при сульфатах 200-300 мг/л',
    description='Запрос:', style=st, layout=widgets.Layout(width='60%'))
geo_dd  = widgets.Dropdown(options=geo_opts,  description='География:',
                            style=st, layout=widgets.Layout(width='200px'))
type_dd = widgets.Dropdown(options=doc_types, description='Тип:',
                            style=st, layout=widgets.Layout(width='200px'))
year_dd = widgets.Dropdown(options=year_opts, description='Год:',
                            style=st, layout=widgets.Layout(width='160px'))

def mk(label, s):
    return widgets.Button(description=label, button_style=s,
                          layout=widgets.Layout(width='155px'))
btn_s  = mk('Найти',          'primary')
btn_l  = mk('Ответ LLM',      'success')
btn_c  = mk('РФ vs Мир',      'info')
btn_g  = mk('Пробелы',        'warning')
btn_gr = mk('Граф (весь)',     '')
btn_gq = mk('Граф (запрос)',  '')
out = widgets.Output()

def get_filters():
    geo = None if geo_dd.value  == 'Все источники' else geo_dd.value
    dt  = None if type_dd.value == 'Все типы'      else type_dd.value
    yr  = None if year_dd.value == 'Все годы'      else year_dd.value
    return geo, dt, yr

def on_search(b):
    with out:
        clear_output(); display(HTML(CSS))
        q = query_inp.value.strip()
        if not q: print('Введите запрос'); return
        geo, dt, yr = get_filters()
        results = search(q, n_results=6, geography=geo, doc_type=dt, year=yr)
        display(HTML('<div class="stitle">Результаты: "' + q + '"</div>'))
        if results:
            for r in results: display(HTML(render_card(r)))
        else:
            print('Ничего не найдено')

def on_llm(b):
    with out:
        clear_output(); display(HTML(CSS))
        q = query_inp.value.strip()
        if not q: print('Введите запрос'); return
        geo, dt, yr = get_filters()
        display(HTML('<div class="stitle"> Аналитический ответ LLM</div>'))
        ans = answer_question(q, geography=geo, doc_type=dt, year=yr)
        display(HTML('<div class="llm">' + ans + '</div>'))

def on_compare(b):
    with out:
        clear_output(); display(HTML(CSS))
        q = query_inp.value.strip()
        if not q: print('Введите запрос'); return
        display(HTML('<div class="stitle"> РФ vs Мир: "' + q + '"</div>'))
        display(HTML('<div class="cmp">' + compare_geo(q) + '</div>'))

def on_gaps(b):
    with out:
        clear_output(); display(HTML(CSS))
        display(HTML('<div class="stitle"> Детектор пробелов</div>'))
        display(HTML('<div class="gap">' + detect_gaps() + '</div>'))

def on_graph(b):
    with out:
        clear_output()
        display(HTML('<div class="stitle"> Граф — весь корпус</div>'))
        build_graph()

def on_graph_q(b):
    with out:
        clear_output()
        q = query_inp.value.strip()
        if not q: print('Введите запрос'); return
        display(HTML('<div class="stitle"> Граф по запросу: "' + q + '"</div>'))
        build_graph(query=q)

btn_s.on_click(on_search);  btn_l.on_click(on_llm)
btn_c.on_click(on_compare); btn_g.on_click(on_gaps)
btn_gr.on_click(on_graph);  btn_gq.on_click(on_graph_q)

display(HTML(CSS))
display(HTML('<h2 style="color:#cba6f7;font-family:sans-serif;margin-bottom:2px"> Научный клубок — Норникель</h2>'))
display(HTML('<p style="color:#a6adc8;margin-top:0;font-family:sans-serif">Поисково-аналитическая система знаний R&D</p>'))
display(widgets.HBox([query_inp]))
display(widgets.HBox([geo_dd, type_dd, year_dd]))
display(widgets.HBox([btn_s, btn_l, btn_c, btn_g, btn_gr, btn_gq]))
display(out)

In [ ]:
def export_to_docx(query, geography=None, doc_type=None, year=None):
    """
    Создание отчета: поиск + аналитический ответ LLM + источники + пробелы.
    Скачивается автоматически в браузере.
    """
    doc = DocxDocument()

    # Заголовок
    title = doc.add_heading('Научный клубок — Норникель', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    sub = doc.add_paragraph('Аналитический отчет R&D')
    sub.alignment = WD_ALIGN_PARAGRAPH.CENTER
    sub.runs[0].font.color.rgb = RGBColor(0x6B, 0x6B, 0x6B)

    doc.add_paragraph(f'Дата формирования: {datetime.now().strftime("%d.%m.%Y %H:%M")}')
    doc.add_paragraph(f'Запрос: {query}')

    filters = []
    if geography: filters.append(f'география: {geography}')
    if doc_type:  filters.append(f'тип документа: {doc_type}')
    if year:      filters.append(f'год: {year}')
    if filters:
        doc.add_paragraph('Фильтры: ' + ', '.join(filters))

    doc.add_paragraph()

    # Найденные документы
    doc.add_heading('1. Найденные источники', level=1)
    results = search(query, n_results=6, geography=geography,
                     doc_type=doc_type, year=year)

    if results:
        for i, r in enumerate(results, 1):
            doc.add_heading(f'{i}. {r["filename"]}', level=2)
            table = doc.add_table(rows=4, cols=2)
            table.style = 'Table Grid'
            cells = [
                ('Тип документа', r['doc_type']),
                ('Источник',      r['source']),
                ('Год',           r['year']),
                ('География',     r['geography']),
            ]
            for row_idx, (label, value) in enumerate(cells):
                table.cell(row_idx, 0).text = label
                table.cell(row_idx, 1).text = value
                table.cell(row_idx, 0).paragraphs[0].runs[0].bold = True
            doc.add_paragraph()
            snippet = doc.add_paragraph(r['text'][:500] + '...')
            snippet.runs[0].font.size = Pt(9)
            snippet.runs[0].font.color.rgb = RGBColor(0x44, 0x44, 0x44)
            doc.add_paragraph()
    else:
        doc.add_paragraph('По данному запросу источники не найдены.')

    doc.add_page_break()

    # Аналитический ответ LLM
    doc.add_heading('2. Аналитический ответ', level=1)
    doc.add_paragraph(
        'Ответ сформирован на основе найденных источников '
        'с использованием YandexGPT.'
    )
    doc.add_paragraph()

    answer = answer_question(query, geography=geography,
                              doc_type=doc_type, year=year)
    doc.add_paragraph(answer)
    doc.add_page_break()

    # Отечественная vs заруюежная практика
    doc.add_heading('3. Отечественная vs зарубежная практика', level=1)
    comparison = compare_geo(query)
    doc.add_paragraph(comparison)
    doc.add_page_break()

    # Пробелы в базе знаний
    doc.add_heading('4. Пробелы в базе знаний', level=1)
    doc.add_paragraph(
        'Анализ покрытия корпуса документов по ключевым темам отрасли.'
    )
    doc.add_paragraph()
    gaps_text = detect_gaps()
    doc.add_paragraph(gaps_text)

    # Список источников
    doc.add_page_break()
    doc.add_heading('5. Полный список источников', level=1)
    if results:
        for i, r in enumerate(results, 1):
            p = doc.add_paragraph(style='List Number')
            p.add_run(r['filename']).bold = True
            p.add_run(
                f'\n   Тип: {r["doc_type"]} | '
                f'Источник: {r["source"]} | '
                f'Год: {r["year"]} | '
                f'География: {r["geography"]}'
            ).font.size = Pt(9)

    # Сохранение и скачивание
    safe_query = query[:40].replace(' ', '_').replace('/', '-')
    filename = f'nornickel_report_{safe_query}_{datetime.now().strftime("%Y%m%d_%H%M")}.docx'
    doc.save(filename)
    files.download(filename)
    print(f'Отчёт сохранён и скачан: {filename}')
    return filename


#Кнопка экспорта в интерфейсе
btn_export = widgets.Button(
    description='Скачать DOCX',
    button_style='danger',
    layout=widgets.Layout(width='155px')
)
out_export = widgets.Output()

def on_export(b):
    with out_export:
        clear_output()
        q = query_inp.value.strip()
        if not q:
            print('Введите запрос перед экспортом')
            return
        geo, dt, yr = get_filters()
        print('Формируем отчёт, подождите...')
        export_to_docx(q, geography=geo, doc_type=dt, year=yr)

btn_export.on_click(on_export)

display(HTML(CSS))
display(HTML(
    '<h3 style="color:#cba6f7;font-family:sans-serif">Экспорт отчёта в DOCX</h3>'
    '<p style="color:#a6adc8;font-family:sans-serif">'
    'Введите запрос выше, затем нажмите кнопку — '
    'отчёт скачается автоматически.</p>'
))
display(widgets.HBox([btn_export]))
display(out_export)

In [ ]:
# Синонимы, достоверность, противоречия, Markdown

SYNONYMS = {
    # Процессы
    'electrowinning':       'электроэкстракция',
    'heap leaching':        'кучное выщелачивание',
    'fluidized bed':        'печь взвешенной плавки',
    'flash smelting':       'взвешенная плавка',
    'solvent extraction':   'экстракция растворителем',
    'pressure oxidation':   'автоклавное окисление',
    'bioleaching':          'биовыщелачивание',
    'flotation':            'флотация',
    'hydrometallurgy':      'гидрометаллургия',
    'pyrometallurgy':       'пирометаллургия',
    'desalination':         'обессоливание',
    'ion exchange':         'ионный обмен',
    'reverse osmosis':      'обратный осмос',
    'nanofiltration':       'нанофильтрация',
    'nickel':               'никель',
    'copper':               'медь',
    'cobalt':               'кобальт',
    'platinum':             'платина',
    'palladium':            'палладий',
    'gold':                 'золото',
    'silver':               'серебро',
    'pgm':                  'мпг',
    'matte':                'штейн',
    'slag':                 'шлак',
    'sulphate':             'сульфат',
    'sulfate':              'сульфат',
    'chloride':             'хлорид',
    'electrolyte':          'электролит',
    'catholyte':            'католит',
    'anolyte':              'анолит',
    'pvp':                  'пвп',
    'ew tank':              'ванна электроэкстракции',
    'scrubber':             'скруббер',
    'membrane':             'мембрана',
    'diaphragm':            'диафрагма',
    'au':                   'золото',
    'ag':                   'серебро',
    'pt':                   'платина',
    'pd':                   'палладий',
    'ni':                   'никель',
    'cu':                   'медь',
    'co':                   'кобальт',
    'so2':                  'диоксид серы',
}

def normalize_query(query: str) -> str:
    """Заменяет английские термины на русские синонимы перед поиском.
    Использует границы слов чтобы не ломать составные слова."""
    q = query.lower()
    for eng, rus in SYNONYMS.items():
        q = re.sub(r'\b' + re.escape(eng) + r'\b', rus, q)
    return q


# Уровень достоверности
def confidence_level(n_docs: int) -> str:
    """Оценка достоверности по количеству найденных источников."""
    if n_docs >= 8:
        return 'Высокая  (' + str(n_docs) + ' источников)'
    elif n_docs >= 5:
        return 'Хорошая  (' + str(n_docs) + ' источников)'
    elif n_docs >= 3:
        return 'Средняя  (' + str(n_docs) + ' источников)'
    elif n_docs >= 1:
        return 'Низкая   (' + str(n_docs) + ' источника)'
    else:
        return 'Нет данных'


# Извлечение числовых параметров
def extract_numbers(text: str) -> list:
    """
    Находит числовые параметры в тексте, включая диапазоны.
    Примеры: 300 мг/л, 200-300 мг/л, 200–300 мг/л, 750°C, 1.5 м/с
    """
    units = (
        r'мг/л|г/л|мг/дм3|кг/т|%|°C|градус|м/с|т/сут|А/м2|МПа|кПа|pH|мкм'
    )
    # диапазоны: 200-300 мг/л или 200–300 мг/л
    range_pattern = (
        r'(\d+(?:[.,]\d+)?)\s*[-–]\s*(\d+(?:[.,]\d+)?)\s*(' + units + r')'
    )
    # одиночные: 300 мг/л
    single_pattern = (
        r'(\d+(?:[.,]\d+)?)\s*(' + units + r')'
    )
    results = []
    for match in re.finditer(range_pattern, text, re.IGNORECASE):
        results.append(f'{match.group(1)}–{match.group(2)} {match.group(3)}')

    # одиночные ищем только там где нет диапазона
    text_no_ranges = re.sub(range_pattern, '', text, flags=re.IGNORECASE)
    for match in re.finditer(single_pattern, text_no_ranges, re.IGNORECASE):
        results.append(f'{match.group(1)} {match.group(2)}')

    return list(dict.fromkeys(results))[:15]


# Анализ противоречий
def detect_contradictions(query: str) -> str:
    """поиск посредством YandexGPT противоречий между найденными источниками."""
    results = search(normalize_query(query), n_results=6)
    if len(results) < 2:
        return 'Недостаточно источников для анализа противоречий (найден менее 2).'

    context = '\n\n'.join([
        '[' + r['filename'] + ', ' + r['year'] + ', ' + r['geography'] + ']\n' + r['text']
        for r in results
    ])
    prompt = (
        'Ты — аналитик R&D Норникеля.\n'
        'Проанализируй следующие фрагменты документов по теме: ' + query + '\n\n'
        + context + '\n\n'
        'Задача: найди противоречия между источниками.\n'
        'Формат ответа:\n'
        '1. Если противоречия есть — опиши каждое: '
        'какой источник что утверждает и в чём расхождение.\n'
        '2. Если противоречий нет — напиши "Противоречий не обнаружено" '
        'и укажи в чём источники согласны.\n'
        'Язык ответа: русский.'
    )
    return ask_llm(prompt)


# Улучшенный поиск с синонимами и достоверностью
def smart_search(query: str, n_results: int = 6,
                 geography=None, doc_type=None, year=None) -> dict:
    """
    Поиск с нормализацией запроса, оценкой достоверности
    и извлечением числовых параметров из результатов.
    """
    normalized = normalize_query(query)
    results = search(normalized, n_results=n_results,
                     geography=geography, doc_type=doc_type, year=year)

    # Сбор числовых параметров из всех найденных фрагментов
    all_numbers = []
    for r in results:
        nums = extract_numbers(r['text'])
        all_numbers.extend(nums)
    unique_numbers = list(dict.fromkeys(all_numbers))[:15]

    return {
        'results':     results,
        'confidence':  confidence_level(len(results)),
        'normalized':  normalized,
        'parameters':  unique_numbers,
        'n_found':     len(results),
    }


# Экспорт в Markdown
def export_to_markdown(query: str, geography=None,
                       doc_type=None, year=None) -> str:
    """Формирование Markdown-отчета и его скачивание."""
    data    = smart_search(query, geography=geography,
                           doc_type=doc_type, year=year)
    answer  = answer_question(normalize_query(query),
                              geography=geography,
                              doc_type=doc_type, year=year)
    compare = compare_geo(query)

    lines = [
        '# Научный клубок — Норникель',
        '',
        f'**Дата:** {datetime.now().strftime("%d.%m.%Y %H:%M")}  ',
        f'**Запрос:** {query}  ',
        f'**Нормализован:** {data["normalized"]}  ',
        f'**Достоверность:** {data["confidence"]}',
        '',
        '---',
        '',
        '## 1. Найденные источники',
        '',
    ]

    for i, r in enumerate(data['results'], 1):
        lines += [
            f'### {i}. {r["filename"]}',
            f'- **Тип:** {r["doc_type"]}',
            f'- **Источник:** {r["source"]}',
            f'- **Год:** {r["year"]}',
            f'- **География:** {r["geography"]}',
            f'- **Релевантность:** {r["score"]}',
            '',
            f'> {r["text"][:400]}...',
            '',
        ]

    if data['parameters']:
        lines += [
            '## 2. Числовые параметры в найденных документах',
            '',
            ', '.join(data['parameters']),
            '',
        ]

    lines += [
        '## 3. Аналитический ответ (YandexGPT)',
        '',
        answer,
        '',
        '## 4. Отечественная vs зарубежная практика',
        '',
        compare,
        '',
        '## 5. Источники',
        '',
    ]
    for i, r in enumerate(data['results'], 1):
        lines.append(
            f'{i}. **{r["filename"]}** — {r["doc_type"]}, '
            f'{r["source"]}, {r["year"]}, {r["geography"]}'
        )

    md_text = '\n'.join(lines)

    safe = query[:40].replace(' ', '_').replace('/', '-')
    fname = f'nornickel_{safe}_{datetime.now().strftime("%Y%m%d_%H%M")}.md'
    with open(fname, 'w', encoding='utf-8') as f:
        f.write(md_text)
    files.download(fname)
    print(f'Markdown сохранён: {fname}')
    return md_text


# Новые кнопки в интерфейсе
btn_smart   = widgets.Button(description='Умный поиск',
                              button_style='primary',
                              layout=widgets.Layout(width='155px'))
btn_contra  = widgets.Button(description='Противоречия',
                              button_style='warning',
                              layout=widgets.Layout(width='155px'))
btn_md      = widgets.Button(description='Скачать MD',
                              button_style='',
                              layout=widgets.Layout(width='155px'))
out_extra   = widgets.Output()


def on_smart(b):
    with out_extra:
        clear_output()
        display(HTML(CSS))
        q = query_inp.value.strip()
        if not q:
            print('Введите запрос')
            return
        geo, dt, yr = get_filters()
        data = smart_search(q, geography=geo, doc_type=dt, year=yr)

        display(HTML(
            '<div class="stitle">Умный поиск: "' + q + '"</div>'
            '<div class="card">'
            '<div class="ftitle">Нормализованный запрос</div>'
            '<p>' + data['normalized'] + '</p>'
            '<div class="ftitle">Уровень достоверности</div>'
            '<p>' + data['confidence'] + '</p>'
            + (
                '<div class="ftitle">Числовые параметры в документах</div>'
                '<p>' + ' &nbsp;|&nbsp; '.join(data['parameters']) + '</p>'
                if data['parameters'] else ''
            )
            + '</div>'
        ))
        for r in data['results']:
            display(HTML(render_card(r)))


def on_contra(b):
    with out_extra:
        clear_output()
        display(HTML(CSS))
        q = query_inp.value.strip()
        if not q:
            print('Введите запрос')
            return
        display(HTML('<div class="stitle">Анализ противоречий: "' + q + '"</div>'))
        result = detect_contradictions(q)
        display(HTML('<div class="llm">' + result + '</div>'))


def on_md(b):
    with out_extra:
        clear_output()
        q = query_inp.value.strip()
        if not q:
            print('Введите запрос')
            return
        geo, dt, yr = get_filters()
        print('Формируем Markdown-отчёт...')
        export_to_markdown(q, geography=geo, doc_type=dt, year=yr)


btn_smart.on_click(on_smart)
btn_contra.on_click(on_contra)
btn_md.on_click(on_md)

display(HTML(CSS))
display(HTML(
    '<h3 style="color:#cba6f7;font-family:sans-serif">'
    'Дополнительные инструменты</h3>'
))
display(widgets.HBox([btn_smart, btn_contra, btn_md]))
display(out_extra)